In [1]:
import numpy as np
import pandas as pd

from md_Helpers import (
    ThermalizationConfig,
    run_thermalization,
)


# ============================================================
# Scan ranges — endpoints are included
# ============================================================

kT_values = np.round(
    np.arange(0.7, 1.0 + 0.01, 0.02),
    decimals=10,
)

rho_values = np.round(
    np.arange(0.5, 0.8 + 0.01, 0.02),
    decimals=10,
)


# ============================================================
# Core thermalization settings
# ============================================================

n_cells = 30
nsteps = 100_000
dt = 0.002

# HDF5 log interval in simulation steps.
nlog = 1_000

seed = 1



# ============================================================
# Execution controls
# ============================================================

base_notes = (
    "Thermalization grid scan over temperature and density"
)

# If True, immediately stop when any grid point fails.
# If False, record the failure and continue through the grid.
stop_on_error = False


# ============================================================
# Validate the requested grid before starting
# ============================================================

if len(kT_values) == 0:
    raise ValueError("kT_values cannot be empty.")

if len(rho_values) == 0:
    raise ValueError("rho_values cannot be empty.")

if np.any(kT_values <= 0):
    raise ValueError("Every kT value must be positive.")

if np.any(rho_values <= 0):
    raise ValueError("Every density must be positive.")

if n_cells <= 0:
    raise ValueError("n_cells must be positive.")

if nsteps <= 0:
    raise ValueError("nsteps must be positive.")

if nlog <= 0:
    raise ValueError("nlog must be positive.")

if dt <= 0:
    raise ValueError("dt must be positive.")

# The workflow requires at least 41 evolved log points for
# its five terminal phase-analysis frames.
number_of_evolved_logs = int(np.ceil(nsteps / nlog))

if number_of_evolved_logs < 41:
    raise ValueError(
        "This workflow requires at least 41 evolved log points. "
        f"The current settings provide only "
        f"{number_of_evolved_logs}. Increase nsteps or reduce nlog."
    )

number_of_runs = len(kT_values) * len(rho_values)
number_of_particles = 4 * n_cells**3

print("Thermalization scan")
print(f"  kT values:       {kT_values.tolist()}")
print(f"  density values:  {rho_values.tolist()}")
print(f"  grid size:       {number_of_runs} runs")
print(f"  particles/run:   {number_of_particles:,}")
print(f"  steps/run:       {nsteps:,}")
print(f"  dt:              {dt:g}")
print(f"  log period:      {nlog:,}")
print()


# ============================================================
# Run every kT-density combination
# ============================================================

results = []
scan_index = 0

for kT in kT_values:
    for rho in rho_values:
        scan_index += 1

        kT = float(kT)
        rho = float(rho)

        print(
            f"[{scan_index:>2}/{number_of_runs}] "
            f"Starting kT={kT:.3f}, rho={rho:.3f}"
        )

        config = ThermalizationConfig(
            n_fcc_cells=n_cells,
            target_rho=rho,
            nsteps=nsteps,
            kT=kT,
            dt=dt,
            log_period=nlog,
            seed=seed,
        )

        try:
            result = run_thermalization(config)

            created_new = bool(
                result.get("created_new", False)
            )
            skipped = bool(result.get("skipped", False))

            if created_new:
                action = "created"
            elif skipped:
                action = "reused"
            else:
                action = "returned"

            print(
                f"    {action}: "
                f"Run_ID={result['run_id']}, "
                f"status={result.get('status', 'unknown')}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": result["run_id"],
                "action": action,
                "status": result.get("status"),
                "created_new": created_new,
                "skipped_existing": skipped,
                "run_signature": result.get(
                    "run_signature"
                ),
                "error": None,
            })

        except Exception as error:
            print(
                f"    FAILED: "
                f"{type(error).__name__}: {error}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": None,
                "action": "failed",
                "status": "Failed",
                "created_new": False,
                "skipped_existing": False,
                "run_signature": config.run_signature,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            })

            if stop_on_error:
                raise


# ============================================================
# Display the completed scan
# ============================================================

thermalization_scan = (
    pd.DataFrame(results)
    .sort_values(["kT", "rho"])
    .reset_index(drop=True)
)

print()
print("Thermalization scan results:")
display(thermalization_scan)

print()
print("Run-ID grid:")

run_id_grid = thermalization_scan.pivot(
    index="kT",
    columns="rho",
    values="run_id",
)

display(run_id_grid)

failed_runs = thermalization_scan.loc[
    thermalization_scan["action"] == "failed"
]

if failed_runs.empty:
    print(
        f"All {number_of_runs} grid points completed "
        "or reused an existing matching run."
    )
else:
    print(
        f"{len(failed_runs)} of {number_of_runs} "
        "grid points failed:"
    )
    display(
        failed_runs[
            ["kT", "rho", "error"]
        ]
    )

Thermalization scan
  kT values:       [0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
  density values:  [0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8]
  grid size:       256 runs
  particles/run:   108,000
  steps/run:       100,000
  dt:              0.002
  log period:      1,000

[ 1/256] Starting kT=0.700, rho=0.500
    reused: Run_ID=20260914191900, status=Running
[ 2/256] Starting kT=0.700, rho=0.520
    created: Run_ID=20260914191909, status=Complete
[ 3/256] Starting kT=0.700, rho=0.540
    reused: Run_ID=20260914191959, status=Running
[ 4/256] Starting kT=0.700, rho=0.560
    created: Run_ID=20260914192054, status=Complete
[ 5/256] Starting kT=0.700, rho=0.580
    reused: Run_ID=20260914192101, status=Complete
[ 6/256] Starting kT=0.700, rho=0.600
    reused: Run_ID=20260914192200, status=Running
[ 7/256] Starting kT=0.700, rho=0.620
    created: Run_ID=20260914192238, status=Complete


,scan_index,kT,rho,run_id,action,status,created_new,skipped_existing,run_signature,error
0,1,0.7,0.50,20260914191900,reused,Running,False,True,4766e87bda389dcacc37915af7d42c3f62808a3c525b18...,None
1,2,0.7,0.52,20260914191909,created,Complete,True,False,32a26c58d9b2c9e6e968972e1511933a09f5df64a7eaf1...,None
2,3,0.7,0.54,20260914191959,reused,Running,False,True,265b7240e7b4612a957f4fdd0e3c5aefb8366a91f13044...,None
3,4,0.7,0.56,20260914192054,created,Complete,True,False,a960b8dce9a09e8c0e40a7cfaf13ad537efed3090e5d11...,None
4,5,0.7,0.58,20260914192101,reused,Complete,False,True,542fb4a0a0f7555f6218da8e5a15a1532e3d9b2f71bc08...,None
...,...,...,...,...,...,...,...,...,...,...
251,252,1.0,0.72,20260914215127,created,Complete,True,False,167d7f498cd818b5802b96c933e76abceaa96c4ce6f087...,None
252,253,1.0,0.74,20260914215129,reused,Complete,False,True,d206e7c6ac0a1f661cb9809657154a5a02fd3a856cbcb0...,None
253,254,1.0,0.76,20260914215224,reused,Running,False,True,86e62e186d270accfceacb369630b1af920d901fc1c326...,None
254,255,1.0,0.78,20260914215307,created,Complete,True,False,f28678eba10bcf364ada689aa1930548a643d16b561088...,None



Run-ID grid:


rho,0.50,0.52,0.54,0.56,0.58,0.60,0.62,0.64,0.66,0.68,0.70,0.72,0.74,0.76,0.78,0.80
kT,,,,,,,,,,,,,,,,
0.70,20260914191900,20260914191909,20260914191959,20260914192054,20260914192101,20260914192200,20260914192238,20260914192259,20260914192359,20260914192421,20260914192501,20260914192602,20260914192607,20260914192705,20260914192752,20260914192802
0.72,20260914192905,20260914192937,20260914193003,20260914193105,20260914193118,20260914193204,20260914193301,20260914193302,20260914193401,20260914193444,20260914193501,20260914193600,20260914193630,20260914193700,20260914193756,20260914193809
0.74,20260914193854,20260914193954,20260914193956,20260914194055,20260914194136,20260914194154,20260914194253,20260914194320,20260914194350,20260914194448,20260914194501,20260914194548,20260914194647,20260914194648,20260914194745,20260914194826
0.76,20260914194843,20260914194942,20260914195010,20260914195042,20260914195138,20260914195150,20260914195236,20260914195333,20260914195334,20260914195432,20260914195521,20260914195530,20260914195625,20260914195703,20260914195719,20260914195817
0.78,20260914195845,20260914195916,20260914200011,20260914200030,20260914200109,20260914200205,20260914200210,20260914200303,20260914200351,20260914200359,20260914200456,20260914200533,20260914200554,20260914200650,20260914200713,20260914200749
0.80,20260914200847,20260914200858,20260914200946,20260914201038,20260914201043,20260914201141,20260914201219,20260914201237,20260914201335,20260914201401,20260914201433,20260914201527,20260914201543,20260914201621,20260914201718,20260914201725
0.82,20260914201815,20260914201910,20260914201912,20260914202008,20260914202049,20260914202106,20260914202202,20260914202229,20260914202259,20260914202357,20260914202409,20260914202457,20260914202546,20260914202552,20260914202649,20260914202727
0.84,20260914202747,20260914202844,20260914202914,20260914202941,20260914203039,20260914203052,20260914203137,20260914203231,20260914203233,20260914203330,20260914203412,20260914203428,20260914203524,20260914203553,20260914203623,20260914203721
0.86,20260914203738,20260914203821,20260914203917,20260914203920,20260914204018,20260914204100,20260914204116,20260914204215,20260914204242,20260914204315,20260914204410,20260914204424,20260914204508,20260914204605,20260914204608,20260914204712


All 256 grid points completed or reused an existing matching run.
